Grundregeln:
- jede python Klasse hat eine übergeordnete Aufgabe
- jede python Klasse hält nur einen data file
- jede python Klasse importiert nur dann andere Klassen, wenn sie nur durch diese Importe als Gesamtes funktionieren (das ist in den meisten Fällen der DataManager). Wenn eine andere Klasse kein inhärenter Teil ist, wird sie nur als Funktions-Input für die Methode nutzbar gemacht.
- kein cleaning! Statt stillem Bereinigen stoppt die Funktion bei Ungereimtheiten und zeigt eine Fehlermeldung an

Zu adressierende Probleme:
- predicate requisite kleiner Teil der RO-Relationen. Reicht der aus?
- nur Abstracts in pubmed. Volltext-Reviews / Metaanalysen oder Bücher besser?
- manche Attribute sind bei fast jeder Entität TRUE. Sollte man diese entfernen? >90% TRUE?
- mehrere Klassen haben durch das LLM denselben Namen bekommen, was verboten ist.
- bis LLMClient().fill_nan werden nur Attribute ausgefüllt, die TRUE sind. Das ist zu ungenau. Von Anfang an müssen Attribute auch False gelabelt werden.
- causally_related ist oft falsch verknüpft. Aus der Liste der Prädikate entfernen?

Adressierte Probleme:
- Abstracts enthalten sehr viele Entitäten, die nichts mit Wunden zu tun haben. Antwort: LLM prompt wurde so angepasst, dass nur domänenspezifische Entitäten integriert werden.
- Die formale Konzeptanalyse hält sich nicht an die BFO/OBO-Architektur. Antwort: attributes haben die form predicate: target. präformierte attributes sind die bfo_classes als is_a:bfo_class um die Einteilung in die BFo-Architektur zu erzwingen.
- es kann mehrere synonme Objekte oder Attribute geben. Antwort: objects und attributes werden per LLM zu standardbegriffen gemappt, dann grouping durch hard code
- Klassennamen werden aus Attributen gebildet. Antwort: Semantische Klassennamen kreiert LLM
- zu viele concepts und attribute? unschaffbar für hard code? erweiterter RAM und TPU
- zu viele Anfragen an LLM? Antwort: batching
- zu komplexe Anfragen an LLM? Antwort: Lange Frage besser als lange Antwort! (gemini 3 flash schafft 1mio tokens)
- komplette Nachverfolgbarkeit für selektives Löschen query-weise oder datumsweise bei schlechten einzelnen Ergebnissen: Antwort: meta_ids
- kein überladen des Speicher: Antwort: immer nur 10 dateien eines Präfix im Speicher und purge method des DataManager()
- LLM-Anfragen dauern zu lange, teils nur eine Anfrage pro 2,5 Minuten. Antwort: asyncio parallel prompting


# 1. Einrichtung der Grundbausteine

## 1.2 Installieren der benötigten Bibliotheken

## 1.3 Import der benötigten Bibliotheken

In [28]:
import nest_asyncio
nest_asyncio.apply()



In [29]:
import pandas as pd
import numpy as np
import re
import os
import glob
import seaborn as sns
from pydantic import validate_call, ConfigDict
import matplotlib.pyplot as plt
from scipy.stats import kruskal, chi2_contingency
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
#from adjustText import adjust_text
import statsmodels.formula.api as smf

In [30]:
import pymc as pm
import arviz as az
import tabulate
import jax
import time
import json
import google.genai as genai
from concepts import Context
import owlready2
import time
from google.api_core import exceptions
from owlready2 import *
import asyncio
from Bio import Entrez, Medline
from IPython.display import display
import types
from abc import ABC, abstractmethod
import requests
from datetime import datetime
from pathlib import Path
from google.genai import types
from pydantic import BaseModel, Field
import pickle


from google.genai.errors import ServerError

In [31]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Works whether VS Code runs the notebook from project root or notebooks/
_cwd = Path().resolve()
_project_root = _cwd if (_cwd / 'config').exists() else _cwd.parent
sys.path.insert(0, str(_project_root))
load_dotenv(_project_root / '.env')

from config.settings import (
    BASE_PATH,
    OWL_OUTPUT_FILE,
    BFO_OWL_URL,
    RO_OWL_URL,
    ENTREZ_EMAIL,
    NCBI_API_KEY,
    GOOGLE_API_KEY,
    BATCH_SIZE,
    CACHE_MAX_FILES_PER_PREFIX,
    PUBMED_SEARCH_TERMS,
    ALLOWED_PREDICATES,
)
print(f"Project root: {_project_root}")
print(f"ALLOWED_PREDICATES: {ALLOWED_PREDICATES}")

Project root: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology
ALLOWED_PREDICATES: ['is_a', 'part_of', 'has_part', 'has_quality', 'occurs_in', 'has_participant']


# Klassenbibliothek

## class DataManager

In [32]:
class DataManager:
    """
    Handles saving and versioning of wound ontology data to storage.
    """
    def __init__(self, base_path=BASE_PATH):
        self.base_path = Path(base_path)
        # Automatically create the folder if it doesn't exist
        self._ensure_directory()
        self.file_name = "meta_ids"
        self.meta_id_dict = self.load_latest_json(self.file_name) or {}

    def _ensure_directory(self):
        """Creates the storage directory if it's missing."""
        try:
            self.base_path.mkdir(parents=True, exist_ok=True)
            print(f"Directory ready: {self.base_path}")
        except Exception as e:
            print(f"Note: Could not create directory (Local mode?): {e}")

    def _get_timestamped_name(self, prefix, extension):
        """Generates a filename like: wound_data_2026-03-05_21-30.json"""
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
        return f"{prefix}_{timestamp}.{extension}"

    def _cleanup_old_files(self, prefix, limit=10):
        """
        Removes oldest files matching the prefix, regardless of extension.
        Assumes prefix is unique to file type.
        """
        # search for any file starting with prefix_
        search_pattern = str(self.base_path / f"{prefix}_*")
        files = sorted(glob.glob(search_pattern))

        if len(files) > limit:
            # The oldest files are at the start of the sorted list
            files_to_delete = files[:-limit]
            for file_path in files_to_delete:
                try:
                    os.remove(file_path)
                    print(f"Cleanup: Pruned old version {os.path.basename(file_path)}")
                except Exception as e:
                    print(f"Cleanup Warning: Could not delete {file_path}: {e}")


    def save_as_json(self, data_list, filename_prefix):
        """Saves JSON and prunes by prefix."""
        filename = self._get_timestamped_name(filename_prefix, "json")
        full_path = self.base_path / filename

        with open(full_path, 'w', encoding='utf-8') as f:
            json.dump(data_list, f, indent=4)

        self._cleanup_old_files(filename_prefix) # Extension-agnostic
        return str(full_path)

    def save_as_csv(self, df, filename_prefix):
        """Saves CSV and prunes by prefix."""
        filename = self._get_timestamped_name(filename_prefix, "csv")
        full_path = self.base_path / filename

        df.to_csv(full_path, index=True)

        self._cleanup_old_files(filename_prefix) # Extension-agnostic
        return str(full_path)


    def save_as_pickle(self, obj, filename_prefix):
        filename = self._get_timestamped_name(filename_prefix, "pkl")
        full_path = self.base_path / filename
        with open(full_path, 'wb') as f:
            pickle.dump(obj, f)
        return str(full_path)


    def _find_latest_path(self, prefix: str, extension: str):
        """
        Helper: Searches the directory for the most recent file
        matching the prefix and extension.
        """
        search_pattern = str(self.base_path / f"{prefix}_*.{extension}")
        files = glob.glob(search_pattern)

        if not files:
            return None

        # Standard sort puts the latest YYYY-MM-DD_HH-MM at the end [-1]
        return sorted(files)[-1]

    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def load_latest_csv(self, prefix: str):
        """Loads the most recent CSV matching the prefix into a DataFrame."""
        path = self._find_latest_path(prefix, "csv")

        if path:
            print(f"Loading latest CSV: {os.path.basename(path)}")
            return pd.read_csv(path, encoding='utf-8')

        print(f"No CSV found for prefix: {prefix}")
        return None

    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def load_latest_json(self, prefix: str):
        """Loads the most recent JSON matching the prefix into a Dictionary."""
        path = self._find_latest_path(prefix, "json")

        if path:
            print(f"Loading latest JSON: {os.path.basename(path)}")
            with open(path, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"No JSON found for prefix: {prefix}")
        return None

    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def load_latest_pickle(self, prefix: str):
        """Loads the most recent Pickle matching the prefix into an Object."""
        path = self._find_latest_path(prefix, "pkl")

        if path:
            print(f"Loading latest Pickle: {os.path.basename(path)}")
            try:
                with open(path, 'rb') as f:
                    return pickle.load(f)
            except Exception as e:
                print(f"Error loading pickle: {e}")
                return None

        print(f"No Pickle found for prefix: {prefix}")
        return None


    def get_meta_stamp(self, workflow_prefix: str):
        """
        Returns ONLY the metadata fields for a specific workflow.
        Returns: dict e.g., {'meta_extract_id': 'extract_1_20260316_1200'}
        """
        # 1. Increment the persistent state
        self.meta_id_dict[workflow_prefix] = self.meta_id_dict.get(workflow_prefix, 0) + 1

        # 2. Persist the state immediately
        self.save_as_json(self.meta_id_dict, self.file_name)

        # 3. Format the unique ID
        count = self.meta_id_dict[workflow_prefix]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        meta_id=f"{workflow_prefix}_{count}_{timestamp}"
        meta_col_name = f"meta_{workflow_prefix}_id"

        return meta_col_name, meta_id

    def purge_by_prefix(self, prefix: str, confirm: bool = False):
        """
        Deletes ALL files matching the prefix (json, csv, pkl).
        Safety: Requires confirm=True to actually execute deletion.
        """
        if not prefix or len(prefix) < 5:
            print(f"Purge aborted: Prefix '{prefix}' is too short/dangerous.")
            return

        # Find all files starting with prefix_
        search_pattern = str(self.base_path / f"{prefix}_*")
        files_to_purge = glob.glob(search_pattern)

        if not files_to_purge:
            print(f"Purge: No files found matching prefix '{prefix}'.")
            return

        print(f"--- PURGE WARNING: Found {len(files_to_purge)} files for '{prefix}' ---")
        for f in files_to_purge:
            print(f"Target: {os.path.basename(f)}")

        if not confirm:
            print("Purge: Action cancelled. Set confirm=True to delete.")
            return

        # Execution
        count = 0
        for file_path in files_to_purge:
            try:
                os.remove(file_path)
                count += 1
            except Exception as e:
                print(f"Purge Error: Could not delete {file_path}: {e}")

        print(f"Purge Complete: Deleted {count} files matching '{prefix}'.")

### proof of function

In [33]:
data_manager=DataManager()

Directory ready: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology/data/cache
Loading latest JSON: meta_ids_2026-04-08_23-00.json


In [34]:
#data_manager.load_latest_json("list_of_ulms_labels")

In [35]:
#data_manager.purge_by_prefix("context_matrix",confirm=True)

## Connectors

### class BaseConnector

In [36]:
class BaseConnector(ABC):
    """
    Abstract Base Class for all data and ontology sources.
    Enforces a consistent interface for the Orchestrator.
    """

    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def __init__(self, source_name: str, data_manager: DataManager, file_name: str):
        self.source_name = source_name
        print(f"Initializing {self.source_name}...")
        self.dm = data_manager
        self.file_name = file_name

    @abstractmethod
    def connect(self):
        """Establish connection or load local resources."""
        pass

    @abstractmethod
    def get_data(self, query):
        """Main method to retrieve information."""
        pass

    @abstractmethod
    def save(self):
        pass

### class OntoConnector

In [37]:
class OntoConnector(BaseConnector):
    def __init__(self, name, data_manager, filename, onto_url):
        super().__init__(name, data_manager, filename)
        self.onto_url = onto_url
        self.onto = None
        self.label_to_iri = {}
        self.iri_to_label = {}

        # Load existing labels
        loaded = self.dm.load_latest_json(self.file_name)
        self._label_list = loaded if isinstance(loaded, list) else []
        self.connect()

    @property
    def label_list(self):
        if not self._label_list:
            self.fetch_labels()
        return self._label_list

    @label_list.setter
    def label_list(self, new_labels):
        if not isinstance(new_labels, list):
            new_labels = [new_labels]
        combined = set(self._label_list) | set(new_labels)
        self._label_list = sorted(list(combined))

    def connect(self):
        try:
            from owlready2 import get_ontology
            print(f"Loading {self.source_name} from {self.onto_url}...")
            self.onto = get_ontology(self.onto_url).load()
        except Exception as e:
            print(f"Failed to load ontology: {e}")

    def get_data(self, query=None):
        """Implementation of ABC: Returns the full label list."""
        return self.label_list

    def fetch_labels(self):
        """To be implemented by BFO or RO subclasses."""
        pass

    def save(self):
        self.dm.save_as_json(self.label_list, self.file_name)

#### class BFOConnector

In [38]:
class BFOConnector(OntoConnector):
    def __init__(self, data_manager):
        super().__init__("BFO Ontology",
            data_manager,
            "bfo_class_labels",
            BFO_OWL_URL
        )

    def fetch_labels(self):
        if self.onto:
            labels = []
            for cls in self.onto.classes():
                # Prefer label, fallback to name
                label = cls.label[0].lower() if cls.label else cls.name
                self.label_to_iri[label] = cls.iri
                self.iri_to_label[cls.iri] = label
                labels.append(label)
            self.label_list = labels

##### Proof of function

In [39]:
#try_bfo_onto_conn=BFOConnector(data_manager)
#try_bfo_onto_conn.get_data()

In [40]:
#try_bfo_onto_conn.save()

#### class ROConnector

In [41]:
class ROConnector(OntoConnector):
    def __init__(self, data_manager):
        super().__init__("Relations Ontology",
            data_manager,
            "ro_property_labels",
            RO_OWL_URL
        )

    def fetch_bfo_relations(self):
        """
        Fetches only foundational properties originally defined in BFO.
        These are the most robust predicates for Formal Concept Analysis.
        """
        if not self.onto:
            return []

        bfo_labels = []
        for prop in self.onto.object_properties():
            # Check if the IRI is a BFO identifier
            if "BFO_" in prop.iri:
                label = (prop.label[0].lower() if prop.label else prop.name).replace(" ", "_")

                # We still keep our label_to_iri mapping for the OWL Exporter
                self.label_to_iri[label] = prop.iri
                self.iri_to_label[prop.iri] = label

                bfo_labels.append(label)

        # BFO is small (usually < 40 relations), so no strict limit needed
        return sorted(list(set(bfo_labels)))

    def fetch_labels(self):
        if self.onto:
            labels = []
            # We look at object_properties() instead of classes()
            for prop in self.onto.object_properties():
                label = prop.label[0].lower() if prop.label else prop.name
                # Sanitize for LLM (e.g., "part of" -> "part_of")
                clean_label = label.replace(" ", "_")

                self.label_to_iri[clean_label] = prop.iri
                self.iri_to_label[prop.iri] = clean_label
                labels.append(clean_label)
            self.label_list = labels

##### Proof of function

### class ULMSConnector

In [42]:
class UMLSConnector(BaseConnector):
    def __init__(self, data_manager,api_key):
        super().__init__("UMLS API", data_manager,"list_of_ulms_labels")
        self.api_key = api_key
        self.base_url = "https://uts-ws.nlm.nih.gov/rest"
        # NLM allows 20 requests/sec with an API key.
        # 0.06s sleep = ~16 requests/sec (Safe side).
        self.delay = 0.06
        self._list_of_labels=self.dm.load_latest_json("list_of_ulms_labels")

    def connect(self):
        """Verifies API key presence."""
        return self.api_key is not None

    @property
    def list_of_labels(self):
        """Getter: Returns the current list of unique labels."""
        return self._list_of_labels

    @list_of_labels.setter
    def list_of_labels(self, new_labels):
        """
        Setter: Adds new labels to the existing list,
        ensuring uniqueness and avoiding overwrites.
        """
        if not isinstance(new_labels, list):
            new_labels = [new_labels]

        # Combine existing and new, then cast to set for uniqueness, then back to list
        combined = set(self._list_of_labels) | set(new_labels)
        self._list_of_labels = list(set(combined))
        self.save()

    def get_data(self, term):
        """
        Fetches new labels for a term and adds them to the existing collection
        without deleting previous data.
        """
        print(f"Searching UMLS for: {term}...")
        result_cuis = self._get_cui_paginated(term)

        new_found_labels = []
        total = len(result_cuis)

        for i, cui in enumerate(result_cuis):
            label = self._get_label_from_cui(cui)
            if label:
                new_found_labels.append(label)

            time.sleep(self.delay)

            if i % 50 == 0:
                print(f"  Processed {i}/{total} concepts...")

        # Use the setter to merge data
        self.list_of_labels = new_found_labels

        print(f"Update complete. Total unique labels now: {len(self.list_of_labels)}")
        return self.list_of_labels


    def save(self):
        self.dm.save_as_json(self.list_of_labels,self.file_name)

    def _get_cui_paginated(self, term):
        """Fetches ALL matching CUIs using 250-item pages."""
        all_cuis = []
        page_number = 1
        page_size = 250

        while page_number<6:
            endpoint = f"{self.base_url}/search/current"
            params = {
                "string": term,
                "apiKey": self.api_key,
                "returnIdType": "concept",
                "pageSize": page_size,
                "pageNumber": page_number
            }

            try:
                res = requests.get(endpoint, params=params)
                if res.status_code == 200:
                    data = res.json().get("result", {})
                    results = data.get("results", [])

                    if not results:
                        break

                    page_cuis = [item["ui"] for item in results if item["ui"] != "NONE"]
                    all_cuis.extend(page_cuis)

                    # If the page isn't full, there are no more pages
                    if len(results) < page_size:
                        break

                    page_number += 1
                    time.sleep(self.delay) # Respect rate limit during search too
                else:
                    print(f"Search Error {res.status_code} at page {page_number}")
                    break
            except Exception as e:
                print(f"Connection error: {e}")
                break

        return all_cuis

    def _get_label_from_cui(self, cui):
        """Fetches the official Preferred Name for a CUI."""
        endpoint = f"{self.base_url}/content/current/CUI/{cui}"
        params = {"apiKey": self.api_key}

        try:
            res = requests.get(endpoint, params=params)
            if res.status_code == 200:
                return res.json().get("result", {}).get("name")
        except:
            return None
        return None

#### Proof of function

In [43]:
#try_ulms_conn=UMLSConnector(data_manager,"ec3add19-51b3-49d6-866c-4233f3cd5f8a")

In [44]:
#try_ulms_conn.get_data("stab wound")

In [45]:
#try_ulms_conn.save()

### class PubmedConnector

In [46]:
class PubmedConnector(BaseConnector):
    """
    Connector for PubMed (NCBI Entrez) to fetch medical abstracts.
    Maintains a local CSV database to avoid redundant API calls.
    """
    def __init__(self, data_manager, email):
        super().__init__("PubMed API", data_manager,"pubmed_abstracts")
        self.email = email
        self._database = self.dm.load_latest_csv(self.file_name)
        self.connect()

    def connect(self):
        """Implementation of ABC: Configures Entrez credentials."""
        if not self.email:
            raise ValueError("PubMed Connector requires a valid email for Entrez.")
        Entrez.email = self.email
        print(f"Connected to PubMed via {self.email}")

    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def get_data(self, query:str, max_abstracts=10):
        """
        Input: query=str, max_abstracts=int
        Output: DataFrame with columns: pmid, title, abstract
        """
        self.database = self._fetch_abstracts(query,max_abstracts)
        return self.database


    def save(self):
        self.dm.save_as_csv(self.database,self.file_name)


    @property
    def database(self):
        """Truly lazy loader: No predefined columns."""
        if self._database is None:
            # If load fails or file is missing, return a blank slate
            self._database = pd.DataFrame()
        return self._database


    @database.setter
    def database(self, new_data: pd.DataFrame):
        """
        Updates the existing database with new_data instead of overwriting it.
        """
        # 1. Initialize the current database if it's still None
        current_df = self.database

        if not isinstance(new_data, pd.DataFrame):
            raise TypeError("New data must be a Pandas DataFrame.")

        # 2. Combine the old and new
        # ignore_index=True prevents duplicate index numbers (0, 1, 0, 1...)
        updated_df = pd.concat([current_df, new_data], ignore_index=True)

        # 3. Optional: Remove exact duplicates if necessary
        self._database = updated_df.drop_duplicates().reset_index(drop=True)
        self.save()
        print(f"Database updated. New total rows: {len(self._database)}")



    def _fetch_abstracts(self, query, max_abstracts):
        """Search and retrieve Medline records, then sync with local storage."""
        try:
            # 1. Search for pmids
            with Entrez.esearch(db="pubmed", term=query, retmax=max_abstracts) as handle:
                id_list = Entrez.read(handle)["IdList"]

            if not id_list:
                print("No abstracts found for the given query.")
                return self.database # Return existing DB if nothing new found

            # 2. Fetch and Parse Medline details
            with Entrez.efetch(db="pubmed", id=id_list, rettype="medline", retmode="text") as handle:
                  records = list(Medline.parse(handle))

            # 3. Transform to DataFrame
            # pmid as a string!!

            meta_col_name, meta_id = self.dm.get_meta_stamp("abstract_query")

            new_data = [{
            "pmid": str(r.get("PMID")),
            meta_col_name: meta_id,  # <--- Tracking tag
            "query": query,      # <--- Keep track of which search found this
            "title": r.get("TI"),
            "abstract": r.get("AB")
    }
    for r in records if "AB" in r
]
            df_new = pd.DataFrame(new_data)

            self.database = df_new

            return self.database

        except Exception as e:
             print(f"Error during fetch: {e}")
             return self.database

#### Proof of function

In [47]:
#try_pub_conn=PubmedConnector(data_manager, "moritz.milewski@master-bids.de")

In [48]:
#try_pub_conn.get_data("wound classification ICNP")

In [49]:
#try_pub_conn.save()

## LLMClient and helper

### pydantic helper classes

In [50]:
# 1. Nested schema to enforce RO/BFO Triple structure
class FCAAttribute(BaseModel):
    predicate: str  # e.g., "is_a", "has_part"
    target: str     # e.g., "CDC Surgical Wound Classification"

class Entity(BaseModel):
    object: str
    true_attributes: list[FCAAttribute] = Field(
        description="Attributes explicitly confirmed in the text."
    )
    false_attributes: list[FCAAttribute] = Field(
        description="Attributes explicitly denied or ruled out in the text."
    )

class ExtractResponse(BaseModel):
    entities: list[Entity]


class NormalizationEntry(BaseModel):
    original: str
    standard: str

class NormalizationResponse(BaseModel):
    mappings: list[NormalizationEntry]


class FillNanResponse(BaseModel):
    confirmed_attributes: list[str]


class SemanticNameResponse(BaseModel):
    superclass_name: str

### class LLMClient

In [51]:
class LLMClient:
    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)
        self.model_id = "gemini-3-flash-preview"

    async def _call_api(self, prompt, response_model, temperature=0.1):
        return await self.client.aio.models.generate_content(
            model=self.model_id,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=response_model,
                temperature=temperature
            )
        )

    async def _execute_with_retry(self, operation_func, *args, **kwargs):
        max_retries = 5
        for attempt in range(max_retries):
            try:
                # We await the operation here
                return await operation_func(*args, **kwargs)
            except Exception as e: # Catch relevant errors (ServerError, etc.)
                if attempt < max_retries - 1:
                    wait = (attempt + 1) * 5
                    print(f"[LLM] Error. Retry {attempt+1} in {wait}s...")
                    # NON-BLOCKING sleep: allows other tasks to run
                    await asyncio.sleep(wait)
                else:
                    raise e


    async def extract_entities(self, text, predicates, targets, attributes):
        """
        Extracts clinical entities using a strict Pydantic response schema.
        """

        prompt = f"""
        Analyze the following clinical text to build a formal wound ontology compliant with BFO and RO standards. The domain is clinical and surgical (medication, surgical treatment, physical therapy, occupational therapy, etiology, pathophysiology, classifications), not basic science research.

        GOAL: Extract objects and their FORMAL ATTRIBUTES. A formal attribute must have a classifying function (e.g., 'CDC Classification', 'Grade IV'). Avoid purely descriptive details as primary attributes, unless they are defining characteristics of the class.
        Every Attribute is a composite of a predicate and a target.

        CONSTRAINTS:
        1. Use only the following RO derived Predicates: {predicates}
        2. Reuse the following targets already present in the context matrix whenever possible {targets}. Create new ones if necessary. Targets can be adverbs, adjectives or nouns. Nouns are always singular.
        3. Ensure "is_a" relations are used to link specific objects to their superclass.
        4. Add confirmed attributes to the true_attributes list.
        5. Add denied attributes to the false_attributes list.
        6. These are the attributes already in use: {attributes}. Try classifying every attribute already in use into true or false and add to the respective list.


        EXAMPLE:
        Object: "Class 4 Wound"
        true_attributes:
          - predicate: "is_a", target: "wound"
          - predicate: "has_quality", target: "Infected"
          - predicate: "has_part", target: "Necrotic Tissue"
          - predicate: "is_member_of", target: "CDC Wound Classification"

        false_attributes:
          - prediate: "is_a", target: "Surgical Instrument"
          - predicate: "has_quality", target: "blue"

        TEXT: {text}
        """

        response = await self._execute_with_retry(self._call_api,prompt, ExtractResponse)

        return response.parsed.entities


    async def create_normalization_mapping(self, terms, standards):
        """
        Maps original medical terms to standardized clinical terms.
        """

        prompt = f"""
        You are provided a batch of medical terms which have to be normalized. Synonyms have to be grouped into one standard term.
        The following input terms have already been found:
        INPUT STANDARDS: {standards}
        TERMS TO PROCESS: {terms}
        STRICT RULES:
        1. If a term is a semantic and word type match for an 'INPUT STANDARD', you MUST use that standard.
        2. Do NOT invent a new term if an existing one suffices.
        3. PRESERVE WORD TYPE: Map adjectives to adjectives (e.g., 'bactericidal' -> 'antibacterial') and nouns to nouns (e.g., 'antibiotic' -> 'anti-bacterial agent'), verbs to verbs etc.
        4. If you must create a new term because NO standard matches the word type and meaning, create a new standard term that matches the Part of Speech of the original term.
        """

        response = await self._execute_with_retry(self._call_api,prompt, NormalizationResponse)

        return {item.original: item.standard for item in response.parsed.mappings}


    async def fill_nan(self, object_name: str, attributes: list[str]) -> list[str]:
        """
        Resolves NaNs by verifying which attributes apply to a given object.
        Used by _Matrix.fill_nan.
        Output: list of confirmed attributes
        """
        if not attributes:
            return []

        prompt = f"""
        Act as a clinical ontologist. Review the Formal Object {object_name}.
        Evaluate the following list of candidate attributes (predicates and targets).

        TASK:
        Return a JSON list of ONLY the attributes that are FACTUALLY and CLINICALLY true for {object_name}.
        If an attribute is incorrect, uncertain, or not a defining characteristic, EXCLUDE it.

        CANDIDATE ATTRIBUTES:
        {attributes}

        Return the results in the original string format provided.
        """
        response = await self._execute_with_retry(self._call_api, prompt, FillNanResponse, temperature=0.0)

        return response.parsed.confirmed_attributes


    async def generate_semantic_name(self, intent_attributes,superclass_names):
        """
        Generates a concise medical superclass name.
        """
        prompt = f"""
        These medical attributes describe a class: {list(intent_attributes)}
        These are the superclass names already found: {superclass_names}
        Generate ONE short medical superclass name (max 5 words, noun only). Be explicit enough to avoid any duplicates.
        """

        response = await self._execute_with_retry(self._call_api,prompt, SemanticNameResponse, temperature=0.3)


        return response.parsed.superclass_name

    def close(self):
        self.client.close()

### Proof of function

In [52]:
llm_client=LLMClient(GOOGLE_API_KEY)

In [53]:
task=llm_client.extract_entities("""A wound forms when biological tissues like skin, mucous membranes, and organs are damaged. Different injuries can cause wounds; properly cleaning and dressing the wounds is essential to prevent infections and additional harm.[1][2] The surgical wound classification (SWC) system was initially developed in 1964 by the National Academy of Sciences and the National Research Council. The SWC system was created to represent the bacterial load in a surgical field. The Centers for Disease Control and Prevention (CDC) later refined this system by establishing 4 different classes of wound statuses outlined below.[3] Each class has a postoperative risk of a surgical site infection (SSI) with scores of 1% to 5%, 3% to 11%, 10% to 17%, and more than 27%, respectively. Class 1 wounds are categorized as clean wounds. These types of wounds are not infected, do not exhibit any signs of inflammation, and are typically closed. If drainage is required, a closed draining approach is recommended. It is worth noting that Class 1 wounds do not involve the respiratory, alimentary, genital, or urinary tracts. Examples of clean wounds include an inguinal hernia repair or a thyroidectomy.
#Class 2 wounds are categorized as clean-contaminated, which means they have a low level of contamination. These types of wounds involve entry into the respiratory, alimentary, genital, or urinary tracts but only under controlled circumstances.
#Class 3 wounds are classified as contaminated and typically result from a breach in sterile techniques or leakage from the gastrointestinal tract. Incisions resulting from acute or nonpurulent inflammation are also considered Class 3 wounds.
#Class 4 wounds are considered to be dirty or infected. These injuries usually occur from inadequate treatment of traumatic wounds, gross purulence, and evident infections. When tissues lose vitality, it can lead to Class 4 wounds. This is often caused by surgery or microorganisms found in perforated organs.[3]""", ALLOWED_PREDICATES,[],[])

/var/folders/mx/nh4b04cn5sldd9fv5rcmyk1c0000gp/T/ipykernel_28085/3411648951.py:1: RuntimeWarning: coroutine 'LLMClient.extract_entities' was never awaited
  task=llm_client.extract_entities("""A wound forms when biological tissues like skin, mucous membranes, and organs are damaged. Different injuries can cause wounds; properly cleaning and dressing the wounds is essential to prevent infections and additional harm.[1][2] The surgical wound classification (SWC) system was initially developed in 1964 by the National Academy of Sciences and the National Research Council. The SWC system was created to represent the bacterial load in a surgical field. The Centers for Disease Control and Prevention (CDC) later refined this system by establishing 4 different classes of wound statuses outlined below.[3] Each class has a postoperative risk of a surgical site infection (SSI) with scores of 1% to 5%, 3% to 11%, 10% to 17%, and more than 27%, respectively. Class 1 wounds are categorized as clean w

In [54]:
import asyncio

# Assuming LLMClient is already imported/defined
llm_client = LLMClient(GOOGLE_API_KEY)

async def process_task():
    # Define the semaphore to limit concurrent requests
    sem = asyncio.Semaphore(10)

    text_input = """A wound forms when biological tissues like skin, mucous membranes, and organs are damaged. Different injuries can cause wounds; properly cleaning and dressing the wounds is essential to prevent infections and additional harm.[1][2] The surgical wound classification (SWC) system was initially developed in 1964 by the National Academy of Sciences and the National Research Council. The SWC system was created to represent the bacterial load in a surgical field. The Centers for Disease Control and Prevention (CDC) later refined this system by establishing 4 different classes of wound statuses outlined below.[3] Each class has a postoperative risk of a surgical site infection (SSI) with scores of 1% to 5%, 3% to 11%, 10% to 17%, and more than 27%, respectively. Class 1 wounds are categorized as clean wounds. These types of wounds are not infected, do not exhibit any signs of inflammation, and are typically closed. If drainage is required, a closed draining approach is recommended. It is worth noting that Class 1 wounds do not involve the respiratory, alimentary, genital, or urinary tracts. Examples of clean wounds include an inguinal hernia repair or a thyroidectomy.
#Class 2 wounds are categorized as clean-contaminated, which means they have a low level of contamination. These types of wounds involve entry into the respiratory, alimentary, genital, or urinary tracts but only under controlled circumstances.
#Class 3 wounds are classified as contaminated and typically result from a breach in sterile techniques or leakage from the gastrointestinal tract. Incisions resulting from acute or nonpurulent inflammation are also considered Class 3 wounds.
#Class 4 wounds are considered to be dirty or infected. These injuries usually occur from inadequate treatment of traumatic wounds, gross purulence, and evident infections. When tissues lose vitality, it can lead to Class 4 wounds. This is often caused by surgery or microorganisms found in perforated organs.[3]""" # Your text here
    relations = ALLOWED_PREDICATES

    async with sem:
        # You must 'await' the extraction if it's an async method
        task = await llm_client.extract_entities(text_input, relations, [], [])
        return task

# To run the script:
if __name__ == "__main__":
    result = asyncio.run(process_task())
    print(result)

[Entity(object='Wound', true_attributes=[FCAAttribute(predicate='is_a', target='Tissue Damage'), FCAAttribute(predicate='has_participant', target='Skin'), FCAAttribute(predicate='has_participant', target='Mucous Membrane'), FCAAttribute(predicate='has_participant', target='Organ')], false_attributes=[]), Entity(object='Class 1 Wound', true_attributes=[FCAAttribute(predicate='is_a', target='Wound'), FCAAttribute(predicate='has_quality', target='Clean'), FCAAttribute(predicate='has_quality', target='Closed')], false_attributes=[FCAAttribute(predicate='has_quality', target='Infected'), FCAAttribute(predicate='has_quality', target='Inflamed'), FCAAttribute(predicate='occurs_in', target='Respiratory Tract'), FCAAttribute(predicate='occurs_in', target='Alimentary Tract'), FCAAttribute(predicate='occurs_in', target='Genital Tract'), FCAAttribute(predicate='occurs_in', target='Urinary Tract')]), Entity(object='Class 2 Wound', true_attributes=[FCAAttribute(predicate='is_a', target='Wound'), FCA

## Formal Context and helper classes


### helper class _Matrix

In [55]:
class _Matrix:

    def __init__(self, data_manager, file_name):
        self.dm = data_manager
        self.file_name = file_name
        self._df=self.dm.load_latest_csv(self.file_name)



    @property
    def df(self):
        if self._df is None:
            self._df = pd.DataFrame()
        return self._df


    @df.setter
    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def df(self, new_data: pd.DataFrame):
        """Adds new data and triggers a global merge."""
        # Check for None OR Empty DataFrame
        if self._df is None or self._df.empty:
            combined = new_data
        else:
            combined = pd.concat([self._df, new_data], sort=False)

        # Apply logic and persist
        self._df = self.merge(combined)
        self.save()


    def merge(self, df: pd.DataFrame) -> pd.DataFrame:
        """Standardizes, sanitizes, and collapses mixed-type data safely."""
        if df is None or df.empty:
            return df

        # 1. Standardize Casings (This naturally creates duplicate column names)
        df.index = df.index.astype(str).str.strip().str.title()
        df.columns = df.columns.astype(str).str.strip().str.lower()

        # 2. Strict Separation: Create two independent DataFrames
        is_meta = df.columns.str.startswith("meta_")
        attr_df = df.loc[:, ~is_meta].copy()
        meta_df = df.loc[:, is_meta].copy()

        # 3. Process Attributes (Strictly Numeric logic)
        if not attr_df.empty:
            # Force numeric, handle errors
            # No Nan fill here!!!
            attr_df = attr_df.apply(lambda x: pd.to_numeric(x, errors='coerce'))
            # Collapse duplicate columns (e.g., 'treats: cancer' + 'treats: cancer')
            attr_df = attr_df.T.groupby(level=0).max().T
            # Collapse duplicate rows (e.g., 'Aspirin' + 'Aspirin')
            attr_df = attr_df.groupby(level=0).max()

        # 4. Process Metadata (Strictly String logic)
        if not meta_df.empty:
            # Collapse duplicate columns (just in case)
            meta_df = meta_df.T.groupby(level=0).first().T
            # Collapse duplicate rows (Keep the first metadata entry encountered)
            meta_df = meta_df.groupby(level=0).first()

        # 5. The Final Recombination
        if not meta_df.empty and not attr_df.empty:
            return pd.concat([attr_df, meta_df], axis=1)
        elif not attr_df.empty:
            return attr_df
        else:
            return meta_df



    @property
    def objects(self):
        """Returns list of row names (Formal Objects)."""
        return list(self.df.index)


    @property
    def attributes(self):
        """
        Returns list of column names (Formal Attributes),
        excluding technical metadata columns.
        """
        # Filter out anything starting with 'meta_'
        return [col for col in self.df.columns if not col.startswith("meta_")]


    @property
    def bool_tuples(self):
        """
        Optimized: Returns binary relations only for non-meta attributes.
        """
        # Using .loc is the 'Pandas-native' way to slice columns efficiently
        return [tuple(row) for row in self.df.loc[:, self.attributes].values]

    @property
    def predicates(self) -> list[str]:
        """
        Returns predicates for non-metadata attributes.
        Format: 'Predicate' from 'Predicate: Target'
        """
        # Logic is now applied only to self.attributes
        return [col.split(':', 1)[0].strip() if ':' in col else "" for col in self.attributes]

    @property
    def targets(self) -> list[str]:
        """
        Returns targets for non-metadata attributes.
        Format: 'Target' from 'Predicate: Target'
        """
        # Logic is now applied only to self.attributes
        return [col.split(':', 1)[1].strip() if ':' in col else col for col in self.attributes]



    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def map_normalize(self, target_mapping: dict, object_mapping: dict):
        """
        Normalizes medical attributes while preserving metadata columns.
        """
        # 1. Create a mapping only for the valid attributes
        attr_map = {}

        # We zip the filtered properties to create the new names
        for old_attr, pred, targ in zip(self.attributes, self.predicates, self.targets):
            standard_target = target_mapping.get(targ, targ)
            attr_map[old_attr] = f"{pred}: {standard_target}"

        # 2. Rename the dataframe
        # rename() will ignore columns not in the attr_map (keeping meta_ columns intact!)
        new_df = self.df.rename(columns=attr_map, index=object_mapping or {})

        # 3. Trigger the OVERWRITE!!
        self._df = self.merge(new_df)

        print(f"Normalization applied. Columns normalized: {len(attr_map)}")
        print(f"New matrix shape: {self.df.shape}")



    async def fill_nan(self, llm, batch_size=10):
        """
        Async version: Evaluates all NaNs in parallel waves.
        batch_size: How many objects (rows) to process at the exact same time.
        """
        valid_attributes = self.attributes
        total_nan = self.df[valid_attributes].isna().sum().sum()

        if total_nan == 0:
            print("No medical NaNs found.")
            return self.df

        print(f"🚀 Starting parallel scrutiny for {total_nan} NaNs across {len(self.df)} objects...")

        # 1. Semaphore: The 'Bouncer' that limits us to 10 simultaneous rows
        sem = asyncio.Semaphore(batch_size)

        async def process_row(object_name):
            async with sem:
                # Find NaNs specifically for this row
                row = self.df.loc[object_name, valid_attributes]
                missing = row[row.isna()].index.tolist()

                if not missing:
                    return object_name, []

                # We send the WHOLE list of missing attributes for this row in one go.
                # Gemini 3 Flash can handle 50-100 attributes easily.
                confirmed = await llm.fill_nan(object_name, missing)
                return object_name, confirmed, missing

        # 2. Create the "Task List"
        tasks = [process_row(obj) for obj in self.df.index]

        # 3. Fire everything! (asyncio.gather waits for the whole wave)
        results = await asyncio.gather(*tasks)

        # 4. Update the local copy
        current_df = self.df.copy()
        for res in results:
            if res is None: continue
            obj_name, confirmed_list, original_missing = res

            # Fill all targeted NaNs for this row
            for attr in original_missing:
                current_df.at[obj_name, attr] = 1 if attr in confirmed_list else 0

        # 5. Metadata and Persistence
        meta_col, meta_id = self.dm.get_meta_stamp("llm_fill_nan")
        current_df[meta_col] = meta_id

        self._df = self.merge(current_df)
        self.save()

        print(f"✅ Parallel evaluation complete. Stamp: {meta_id}")
        return self.df

    def save(self):
        """Saves the matrix directly using the DataManager."""
        if not self.df.empty:
            self.dm.save_as_csv(self.df, self.file_name)
        else:
            print("Save skipped: Matrix is currently empty.")

### helper class _Cache

In [56]:
class _Cache:
    """Dictionary-based staging area indexed by pmid."""
    def __init__(self, data_manager, file_name):
        self.dm = data_manager
        self.file_name = file_name
        # Internal dictionary storage: {pmid: {object: [attributes]}}
        self._dict = self.dm.load_latest_json(self.file_name)

    @property
    def dict(self):
        """Access the underlying dictionary."""
        if self._dict is None:
            self._dict = {}
        return self._dict

    @dict.setter
    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def dict(self, new_data: dict):
        """Updates the cache dictionary by merging new pmid entries."""
        self._dict.update(new_data)
        self.save()

    @property
    def attributes(self) -> list[str]:
        """Returns list of unique medical attributes (excluding meta_)."""
        all_attrs = set()
        for pmid_data in self._dict.values():
            for attr_list in pmid_data.values():
                all_attrs.update(attr_list)

        # Filter meta_ columns here just like in the Matrix
        return sorted([a for a in all_attrs if not a.startswith("meta_")])

    @property
    def predicates(self) -> list[str]:
        """Extracted predicates from attributes."""
        return [attr.split(':', 1)[0].strip() if ':' in attr else "" for attr in self.attributes]

    @property
    def targets(self) -> list[str]:
        """Extracted targets from attributes."""
        return [attr.split(':', 1)[1].strip() if ':' in attr else attr for attr in self.attributes]



    async def add_llm_extracts(self, df_pubmed, llm, allowed_predicates, batch_size=10):
        """
        Processes abstracts in parallel waves.
        batch_size: How many abstracts to extract simultaneously.
        """
        meta_col, meta_id = self.dm.get_meta_stamp("llm_extract")
        current_cache = self.dict

        # 1. Identify which PMIDs actually need processing
        to_process = []
        for _, row in df_pubmed.iterrows():
            pmid = str(int(row['pmid']))
            if pmid not in current_cache:
                to_process.append(row)

        if not to_process:
            print("All abstracts already in cache.")
            return

        print(f"🚀 Parallel Extraction: {len(to_process)} new abstracts in waves of {batch_size}...")

        # 2. The Semaphore (The Traffic Controller)
        sem = asyncio.Semaphore(batch_size)

        async def extract_one(row):
            async with sem:
                pmid = str(int(row['pmid']))
                try:
                    # Step 1: Perform the LLM extraction
                    extracted_response = await llm.extract_entities(
                        row['abstract'],
                        allowed_predicates,
                        self.targets,
                        self.attributes
                    )

                    # Step 2: Build the flattened attribute map
                    # Structure: { Entity: { "predicate: target": 1.0 or 0.0 } }
                    entities_map = {}
                    for ent in extracted_response:
                        attr_logic = {}

                        # Map confirmed attributes to 1.0
                        for a in ent.true_attributes:
                            attr_logic[f"{a.predicate}: {a.target}"] = 1.0

                        # Map denied attributes to 0.0
                        for a in ent.false_attributes:
                            triple = f"{a.predicate}: {a.target}"
                            # Priority: If already marked 1.0 in this abstract, keep it 1.0
                            if triple not in attr_logic:
                                attr_logic[triple] = 0.0

                        entities_map[ent.object] = attr_logic

                    # Step 3: Return the PMID as the key
                    return pmid, {
                        "entities": entities_map,
                        "meta_stamp": meta_id
                    }

                except Exception as e:
                    print(f"❌ Failed PMID {pmid}: {e}")
                    return pmid, None

        # 3. Fire the tasks
        tasks = [extract_one(row) for row in to_process]

        # 4. Use as_completed so we can update/save as they finish
        for f in asyncio.as_completed(tasks):
            pmid, result = await f
            if result:
                # Direct update to internal dict
                self._dict[pmid] = result
                # Optional: Save every few results to be safe
                self.save()
                print(f"✅ Cached PMID {pmid}")

        print(f"Batch extraction complete. {len(to_process)} abstracts processed.")


    def save(self):
        """Saves the cache directly using the DataManager (JSON)."""
        if self._dict:
            print("Saving new cache dictionary")
            self.dm.save_as_json(self._dict, self.file_name)
        else:
            print("Save skipped: Cache is empty.")

/opt/anaconda3/envs/first_env/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:648: ArbitraryTypeWarning: <property object at 0x14418bc40> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(


### class FormalContext

In [57]:
class FormalContext:
  def __init__(self, data_manager, llm_client):
        self.dm=data_manager
        self.matrix = _Matrix(self.dm, "context_matrix")
        self.cache = _Cache(self.dm, "extraction_cache")


  def sync_cache_to_matrix(self):
    """
    Adapts the new cache structure:
    { PMID: { "entities": { "EntityName": { "Pred: Target": 1.0 } }, "meta_stamp": ... } }
    """
    flat_map = {}

    for pmid, pmid_data in self.cache.dict.items():
        # Get the 'entities' sub-dict from the PMID entry
        entities_dict = pmid_data.get("entities", {})

        for entity_name, attributes_dict in entities_dict.items():

            if entity_name not in flat_map:
                flat_map[entity_name] = {}

            # attributes_dict is already { "Predicate: Target": 1.0 or 0.0 }
            for attr_triple, value in attributes_dict.items():
                # Validation check: Ensure it's a valid medical triple
                if ":" not in str(attr_triple):
                    continue # Or raise ValueError if you want strictness

                # Priority logic:
                # 1.0 (True) always wins over 0.0 (False) if PMIDs conflict
                current_val = flat_map[entity_name].get(attr_triple, 0.0)
                flat_map[entity_name][attr_triple] = max(current_val, value)

    if not flat_map:
        print("⚠️ Warning: No valid entities found in cache.")
        return

    # 2. Build the list of rows for the DataFrame
    rows = []
    for entity, attributes_dict in flat_map.items():
        row = {"entity": entity}
        row.update(attributes_dict)
        rows.append(row)

    # 3. Create DataFrame and set Index
    # We keep NaNs as NaNs to represent 'Unknown' for your filling step
    new_data = pd.DataFrame(rows).set_index("entity")
    for col in new_data.columns:
        if "meta_" in str(col).lower():
            raise ValueError(f"CRITICAL: Final matrix contains invalid column: '{col}'")

    # 5. Update the DataManager
    self.matrix.df = new_data

    print(f"✅ Sparse Sync Successful (1.0=True, 0.0=False, NaN=Unknown).")
    print(f"   - Unique Entities: {len(new_data)}")
    print(f"   - Medical Triples: {len(new_data.columns)}")

#### Proof of function

In [58]:
#fc=FormalContext(data_manager, LLMClient(GOOGLE_API_KEY))

In [59]:
#fc.sync_cache_to_matrix()

## Mappers

### class BaseMapper

In [60]:
class BaseMapper(ABC):

    def __init__(self, mapper_name: str, data_manager: DataManager, file_name: str):
        self.name = mapper_name
        self.dm = data_manager
        self.file_name = file_name

        self._mapping = self.dm.load_latest_json(self.file_name) or {}

    @property
    def mapping(self) -> dict:
        """Access the underlying mapping dictionary."""
        return self._mapping

    @mapping.setter
    @validate_call(config=ConfigDict(extra='forbid', arbitrary_types_allowed=True))
    def mapping(self, new_data: dict):
        """Updates the mapping and automatically triggers a save."""
        self._mapping.update(new_data)
        self.save()

    def save(self):
        """Persists the mapping state via DataManager."""
        if self._mapping:
            print(f"[{self.name}] Saving updated mapping...")
            self.dm.save_as_json(self._mapping, self.file_name, )

### class NormalizationMapper

In [61]:
class NormalizationMapper(BaseMapper):
    def __init__(self, mapper_name, data_manager: DataManager,file_name):
        super().__init__(
            mapper_name=mapper_name,
            data_manager=data_manager,
            file_name=file_name
        )

    @property
    def standards(self) -> list[str]:
        """Read-only unique list of the 'clean' ontology terms."""
        return sorted(list(set(self._mapping.values())))

    @property
    def originals(self) -> list[str]:
        """Read-only list of all raw terms we have already processed."""
        return sorted(list(self._mapping.keys()))


    async def build_chunkwise_mapping(self, terms: list[str], llm, batch_size=5):
        """
        Async version: Normalizes medical terms in parallel waves.
        batch_size: How many chunks of 30 to process simultaneously.
        """
        # 1. Filter out already processed terms
        terms_to_map = [t for t in terms if t not in self.originals]

        if not terms_to_map:
            print(f"[{self.name}] All {len(terms)} terms found in cache.")
            return self.mapping

        # 2. Chunking (Groups of 30 is still good for context window quality)
        chunks = [terms_to_map[i:i + 30] for i in range(0, len(terms_to_map), 30)]
        print(f"[{self.name}] Found {len(terms_to_map)} new terms. Processing {len(chunks)} async chunks...")

        # 3. Semaphore to avoid overwhelming the API
        sem = asyncio.Semaphore(batch_size)

        async def process_chunk(chunk, chunk_id):
            async with sem:
                # We pass the current standards so the LLM reuses existing terms
                results = await llm.create_normalization_mapping(chunk, self.standards)
                print(f"[{self.name}] Chunk {chunk_id+1} complete.")
                return results

        # 4. Fire the waves
        tasks = [process_chunk(chunk, i) for i, chunk in enumerate(chunks)]

        # as_completed allows us to update the mapping and save as each chunk finishes
        for f in asyncio.as_completed(tasks):
            batch_results = await f
            # This triggers your @mapping.setter (incremental save to Drive)
            self.mapping = batch_results

        print(f"[{self.name}] Normalization complete. Total terms in cache: {len(self.originals)}")
        return self.mapping

#### class TargetMapper

In [62]:
class TargetMapper(NormalizationMapper):
  def __init__(self, data_manager):
      super().__init__(
            mapper_name="Mapper of the target terms in matrix attributes",
            data_manager=data_manager,
            file_name="target_mapping")


#### class ObjectMapper

In [63]:
class ObjectMapper(NormalizationMapper):
  def __init__(self, data_manager):
      super().__init__(
            mapper_name="Mapper of the matrix object names",
            data_manager=data_manager,
            file_name="object_mapping")

## class FormalConceptAnalysis

In [64]:
class FormalConceptAnalysis:

    def __init__(self, data_manager,formal_context):
        self.dm=data_manager
        self.file_name="concept_lattice"
        self.fc = formal_context
        self._lattice = self.dm.load_latest_pickle(self.file_name)

    @property
    def semantic_superclass_names(self):

        """
        Internal logic to filter 'Composite Attributes' out of the hierarchy.
        """
        semantic_names=[]
        all_concepts= self.lattice.concepts
        for concept in all_concepts:

            name = getattr(concept, 'name', None)

            if isinstance(name, str):
                # EXCLUSION LOGIC:
                # If it has a colon, it's a medical triple (Attribute Composite)
                # If it starts with '(', it's a raw tuple
                is_composite = ":" in name or name.startswith("(")

                if not is_composite:
                    semantic_names.append(name)

        return list(semantic_names)


    @property
    def lattice(self):
        if self._lattice is None:
            print("Building lattice...")
            self._lattice = self.build_lattice()
        return self._lattice

    @lattice.setter
    def lattice(self, new_lattice):
        self._lattice = new_lattice
        self.save()

    def save(self):
        self.dm.save_as_pickle(self.lattice,self.file_name)

    def build_lattice(self):
        ctx = Context(self.fc.matrix.objects, self.fc.matrix.attributes, self.fc.matrix.bool_tuples)
        self.lattice = ctx.lattice
        return self.lattice

    def build_lattice(self):
        """
        Purely local calculation.
        Note: We don't make this 'async' because 'Context' is a CPU-bound
        library operation, not a network operation.
        """
        print("🏗️  Constructing Formal Concept Lattice from matrix...")
        # Assuming 'Context' comes from a library like 'concepts' or 'fca'
        from concepts import Context

        ctx = Context(
            self.fc.matrix.objects,
            self.fc.matrix.attributes,
            self.fc.matrix.bool_tuples
        )
        self.lattice = ctx.lattice
        return self.lattice

    async def create_superclass_names(self, llm, min_objects=2, batch_size=10):
        """
        Async version: Generates medical superclass names for concepts in parallel.
        """
        if not self.lattice:
            print("⚠️ No lattice found. Build it first.")
            return None

        # 1. Identify valid concepts that need a name
        concepts_to_name = []
        for concept in self.lattice:
            # Filter logic (Skip top/bottom or tiny/noisy concepts)
            if not concept.intent: continue
            if len(concept.extent) < min_objects: continue

            # If concept already has a custom name, we can skip it (optional)
            concepts_to_name.append(concept)

        print(f"🧬 Found {len(concepts_to_name)} concepts requiring medical naming.")

        # 2. The Semaphore (Traffic Control)
        sem = asyncio.Semaphore(batch_size)

        async def name_worker(concept):
            async with sem:
                try:
                    # Trigger the async LLM method
                    # concept.intent is usually a tuple of strings
                    name = await llm.generate_semantic_name(concept.intent,self.semantic_superclass_names)
                    concept.name = name
                    return True
                except Exception as e:
                    print(f"❌ Failed to name concept {concept.intent[:2]}: {e}")
                    return False

        # 3. Fire the Wave
        tasks = [name_worker(c) for c in concepts_to_name]

        # We use gather to ensure all concepts are named before proceeding
        results = await asyncio.gather(*tasks)

        print(f"✅ Named {sum(results)} superclasses.")
        self.save()
        return self.lattice

## class OWL Exporter

In [65]:
class OWLExporter:
    def __init__(self, data_manager,fca,iri="http://example.org/ontology"):
        self.iri = iri
        self.dm=data_manager
        self.onto = get_ontology(self.iri)
        self.fca=fca

    def create_owl(self, output_file=OWL_OUTPUT_FILE):
        import types as python_types
        with self.onto:
            concept_to_class = {}

            # 1. Schritt: Konzepte als Klassen anlegen
          # Inside your OWLExporter.create_owl loop:

            for concept in self.fca.lattice:
                # 1. Check if the concept has a name assigned by the LLM
                if hasattr(concept, 'name') and concept.name:
                # Standard cleaning: replace spaces with underscores for OWL compatibility
                      cls_name = concept.name.replace(" ", "_").replace("(", "").replace(")", "")
                else:
                      # 2. Handle the "Nameless" concepts (Infimum/Supremum)
                      if not concept.intent and concept.extent:
                          cls_name = "TopMedicalEntity" # The Supremum
                      elif concept.intent and not concept.extent:
                          cls_name = "BottomEmptyConcept" # The Infimum
                      else:
                          # Fallback: Use the attributes if no name exists
                          cls_name = f"Concept_{id(concept)}"



                # Klasse erstellen (erbt standardmäßig von Thing)
                new_class = python_types.new_class(cls_name, (Thing,))
                concept_to_class[concept.index] = new_class

                # 2. Schritt: Objekte aus dem Extent als Blatt-Klassen hinzufügen
                for obj_name in concept.extent:
                    obj_cls_name = obj_name.replace(" ", "_")
                    # Erstellt die Klasse (oder holt sie, falls schon existent)
                    # und fügt die aktuelle Konzept-Klasse als Parent hinzu
                    types.new_class(obj_cls_name, (new_class,))

            # 3. Schritt: Hierarchie zwischen den Konzepten setzen (SubClassOf)
            for concept in self.fca.lattice:
                if concept.index in concept_to_class:
                    cls = concept_to_class[concept.index]
                    for sup in concept.upper_neighbors:
                        if sup.index in concept_to_class:
                            # Das aktuelle Konzept ist eine Unterklasse seines Upper Neighbors
                            cls.is_a.append(concept_to_class[sup.index])

        self.onto.save(file=output_file, format="rdfxml")
        print(f"Reine Klassenhierarchie gespeichert: {output_file}")

## class Orchestrator

In [66]:
class Orchestrator:
    def __init__(self):
        self.dm = DataManager()
        # Initialize the ASYNC Client
        self.llm = LLMClient(GOOGLE_API_KEY)

        self.pubcon = PubmedConnector(self.dm, ENTREZ_EMAIL)
        self.fc = FormalContext(self.dm, self.llm)
        self.fca=FormalConceptAnalysis(self.dm,self.fc)
        self.target_mapper = TargetMapper(self.dm)
        self.object_mapper = ObjectMapper(self.dm)
        self.exporter=OWLExporter(self.dm,self.fca)
        self.allowed_predicates = ALLOWED_PREDICATES

    def _run_sync(self, coroutine):
        """Internal helper to run any async method synchronously."""
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(coroutine)

    def run_pubmed_search(self, query, max_results=50):
        print(f"Searching PubMed for: {query}")
        self.pubcon.get_data(query, max_results)

    def run_extract_to_final_cache(self, batch_size=10):
        print("Starting LLM Extraction...")
        # We wrap the internal async call
        self._run_sync(self.fc.cache.add_llm_extracts(
            self.pubcon.database, self.llm, self.allowed_predicates, batch_size
        ))

    def run_cache_to_nan_matrix(self):
        async def _internal_norm():
            print("Synchronizing Matrix...")
            self.fc.sync_cache_to_matrix()
            print("Creating normalization mappings (Parallel)...")
            await self.target_mapper.build_chunkwise_mapping(self.fc.matrix.targets, self.llm)
            await self.object_mapper.build_chunkwise_mapping(self.fc.matrix.objects, self.llm)
            self.fc.matrix.map_normalize(self.target_mapper.mapping, self.object_mapper.mapping)

        self._run_sync(_internal_norm())

    def run_matrix_nan_fill(self, batch_size=15):
        print(f"Evaluating missing relations (Parallel Waves of {batch_size})...")
        # Wraps the async Matrix.fill_nan
        self._run_sync(self.fc.matrix.fill_nan(self.llm, batch_size=batch_size))

    def run_fca(self):
        async def _internal_fca():
            self.fca.build_lattice()
            await self.fca.create_superclass_names(self.llm)

        self._run_sync(_internal_fca())

    def export_to_owl(self):
        self.exporter.create_owl()

    def run_full_pipeline(self, query, max_results=50):
        self.run_pubmed_search(query, max_results)
        self.run_extract_to_final_cache()
        self.run_cache_to_nan_matrix()
        self.run_matrix_nan_fill()
        self.run_fca()
        self.export_to_owl()    


# Pipeline

In [ ]:
pipeline = Orchestrator()
#pipeline.dm.purge_by_prefix("context_matrix",confirm=True)
#pipeline.run_full_pipeline(PUBMED_SEARCH_TERMS[0])
#pipeline.run_extract_to_final_cache()
pipeline.run_cache_to_nan_matrix()
pipeline.run_matrix_nan_fill()
pipeline.run_fca()
pipeline.export_to_owl()

Directory ready: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology/data/cache
Loading latest JSON: meta_ids_2026-04-08_23-00.json
Initializing PubMed API...
Loading latest CSV: pubmed_abstracts_2026-04-08_23-00.csv
Connected to PubMed via moritz.milewski@master-bids.de
No CSV found for prefix: context_matrix
Loading latest JSON: extraction_cache_2026-04-08_23-15.json
No Pickle found for prefix: concept_lattice
No JSON found for prefix: target_mapping
No JSON found for prefix: object_mapping
Synchronizing Matrix...
✅ Sparse Sync Successful (1.0=True, 0.0=False, NaN=Unknown).
   - Unique Entities: 328
   - Medical Triples: 661
Creating normalization mappings (Parallel)...
[Mapper of the target terms in matrix attributes] Found 653 new terms. Processing 22 async chunks...
[Mapper of the target terms in matrix attributes] Chunk 9 complete.
[Mapper of the target terms in matrix attributes] Saving updated mapping...
[Mapper of the target terms in matrix attributes

In [ ]:
display(pipeline.fc.cache.dict)

In [ ]:
display(pipeline.fc.matrix.df)

In [ ]:
def print_lattice_names(fca_instance):
    print(f"--- 🧬 Exploring {len(fca_instance.lattice)} Lattice Concepts ---")

    named_count = 0
    for i, concept in enumerate(fca_instance.lattice):
        # 1. Check if the LLM actually assigned a name
        # We use getattr to avoid AttributeError if the name wasn't set
        name = getattr(concept, 'name', None)

        if name:
            # 2. Print the Name + some metadata (how many medical objects are in it)
            extent_size = len(concept.extent)
            intent_preview = ", ".join(list(concept.intent)[:3]) # Show first 3 attributes

            print(f"[{i}] Name: {name}")
            print(f"    └─ Extent: {extent_size} objects | Intent: {intent_preview}...")
            named_count += 1

        # Optional: Stop after 50 so you don't scroll forever in Colab
        if named_count >= 50:
            print("\n... and 1,800+ more. Use slicing to see others.")
            break

    print(f"\n✅ Displayed {named_count} named medical superclasses.")

# To run it:
print_lattice_names(pipeline.fca)